<a href="https://colab.research.google.com/github/Rafak22/python/blob/main/M1_Ex1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain langchain-openai pydantic langchain-community -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [2]:
from langchain_core.messages import HumanMessage, SystemMessage

In [3]:
from google.colab import userdata
from langchain_openai import ChatOpenAI

API_Key = userdata.get("API_Key")
BASE_URL = "https://openrouter.ai/api/v1"
MODEL = "nvidia/nemotron-3-nano-30b-a3b:free"

In [4]:
# High temperature (creative / random)
llm_high = ChatOpenAI(
    model=MODEL,
    temperature=1.4,
    api_key=API_Key,
    base_url=BASE_URL,
)

# Low temperature (focused / deterministic)
llm_low = ChatOpenAI(
    model=MODEL,
    temperature=0.1,
    api_key=API_Key,
    base_url=BASE_URL,
)

In [5]:
messages = [
    SystemMessage(content="Be poetic in your response. Reply in JSON with a single key 'answer'. Keep it short."),
    HumanMessage(content="What is artificial intelligence?"),
]

response_high = llm_high.invoke(messages)
response_low  = llm_low.invoke(messages)

print("=== HIGH TEMPERATURE ===")
print(response_high.content)

print("\n=== LOW TEMPERATURE ===")
print(response_low.content)

=== HIGH TEMPERATURE ===


=== LOW TEMPERATURE ===
{
  "answer": "AI is the whisper of thought in silicon, dreaming in patterns."
}


In [6]:
from pydantic import BaseModel
from typing import Literal

class Sentiment(BaseModel):
    sentiment: Literal["positive", "neutral", "negative"]

In [7]:
llm_sentiment = ChatOpenAI(
    model=MODEL,
    temperature=0.0,
    api_key=API_Key,
    base_url=BASE_URL,
)

structured_llm = llm_sentiment.with_structured_output(Sentiment)

In [8]:
from pydantic import BaseModel
from typing import List, Literal

class Categories(BaseModel):
    tags: List[Literal["cars", "shopping", "sports", "study", "work"]]

In [9]:
sentences = [
    "Kindness creates lasting joy.",
    "Success rewards persistent effort.",
    "I love Sunlight. It warms the skin.",
    "Pessemestic all the time.",
    "The storm caused damage!",
    "The clock ticks steadily.",
]

for sentence in sentences:
    result = structured_llm.invoke(sentence)
    print(f"{result.sentiment:10} | {sentence}")

positive   | Kindness creates lasting joy.
positive   | Success rewards persistent effort.
positive   | I love Sunlight. It warms the skin.
positive   | Pessemestic all the time.
neutral    | The storm caused damage!
neutral    | The clock ticks steadily.


In [10]:
!pip install pypdf -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 9.0 MB/s eta 0:00:00


In [11]:
from langchain_community.document_loaders import PyPDFLoader
from pydantic import BaseModel
from typing import List, Optional

class ResumeData(BaseModel):
    candidate_name: str
    skills: List[str]
    experience: List[str]
    education: Optional[str] = None
    email: Optional[str] = None

In [12]:
from google.colab import files
uploaded = files.upload()  # upload your PDF here
filename = list(uploaded.keys())[0]

loader = PyPDFLoader(filename)
pages = loader.load()
cv_text = " ".join([p.page_content for p in pages])

structured_llm_cv = llm_low.with_structured_output(ResumeData)
result = structured_llm_cv.invoke(cv_text)

print("Name      :", result.candidate_name)
print("Email     :", result.email)
print("Education :", result.education)
print("Skills    :", result.skills)
print("Experience:", result.experience)

Saving CV_Rafa Alshareef.pdf to CV_Rafa Alshareef.pdf
Name      : Rafa Alsharif
Email     : rafe.al sharif@email.com (note: email appears to have a typo; correct format needed)
Education : B.S. Computer Science – Artificial Intelligence, Imam Abdulrahman Bin Faisal University (GPA 4.43/5.0), Graduated August 2025
Skills    : ['Programming: Python, Java, C++', 'AI & ML: Machine Learning, NLP, Computer Vision, Agentic AI Systems', 'Frameworks: FastAPI, LangChain', 'Automation: n8n, Make', 'Databases: SQL, Database Management', 'Tools: Git, Linux', 'Soft Skills: Communication, Adaptability, Problem-Solving, Teamwork, Time Management']
Experience: ['AI Specialist | Nexta (Sep 2025 – Jan 2026)', 'Speaker, RAD Exhibition (Nov 2025)', 'Co-op Training, Nexta (Jun – Aug 2025)', 'Graduate Research Assistant, University of Michigan (Oct 2023)', 'Teaching Assistant, University of California, Irvine (Feb 2026)', 'Cloud Computing & AI, Tuwaiq Academy, Alilaab Cloud (Jan 2025)', 'Third Place Award, A

In [13]:
def add(a, b):
    return a + b

def subtract(a, b):
    return a - b

def multiply(a, b):
    return a * b

def divide(a, b):
    return a / b

In [14]:
from pydantic import BaseModel, Field
from typing import Dict, Any

system_prompt = """
You are a math assistant. You have access to these tools:

def add(a, b):
    ...
def subtract(a, b):
    ...
def multiply(a, b):
    ...
def divide(a, b):
    ...

Return the tool name and arguments required to solve the user's request.
"""

class ToolCall(BaseModel):
    tool_name: str = Field(description="The name of the function to use")
    arguments: Dict[str, Any] = Field(description="The parameters to pass")

In [15]:
from langchain_core.messages import SystemMessage, HumanMessage

structured_llm_tools = llm_low.with_structured_output(ToolCall)

question = "what is two plus 5"

response = structured_llm_tools.invoke([
    SystemMessage(content=system_prompt),
    HumanMessage(content=question),
])

print("Tool     :", response.tool_name)
print("Arguments:", response.arguments)

Tool     : add
Arguments: {'a': 2, 'b': 5}


In [16]:
# Execute the tool the model chose
tools_map = {
    "add": add,
    "subtract": subtract,
    "multiply": multiply,
    "divide": divide,
}

fn = tools_map[response.tool_name]
answer = fn(**response.arguments)
print("Answer   :", answer)

Answer   : 7
